# Keltner Squeeze Breakout — Expanding Walk-Forward + Exogenous ML + Hoeffding
**Mario Trevino, FinTech 533, HW5**

Historical backtest (no orders placed). Three-pillar engine:

1. **Signal Alpha** — Keltner Squeeze Breakout with volume (>=1.3×) and ADX(14)>=20 participation filters.
2. **Exogenous Filtering** — Logistic Regression on yield-curve splines, VIX term structure, IV-RV spread, sector relative strength. Strict `fit_transform` on train only.
3. **Statistical Regime Survival** — Egger & Vestal (2025) Hoeffding monitor with N_eff autocorrelation correction and 10% halt line.

**Expanding walk-forward, refit per fold.** At every fold boundary the BQS table, StandardScaler, and Logistic Regression all re-fit on the growing training history. Nothing leaks from future to past.

```
Fold 1:  Train 2021           Test 2022
Fold 2:  Train 2021-2022      Test 2023
Fold 3:  Train 2021-2023      Test 2024
Fold 4:  Train 2021-2024      Test 2025
```

**Universe.** Fifty US equities selected ex-ante by a mechanical rule: "Top 50 by Jan 1, 2021 dollar volume that traded continuously from Jan 1, 2020 through Apr 17, 2026, excluding ETFs and ADRs." Rule decided before any OOS result was viewed. Strategy params (BB lookback, ATR mults, exits, sizing, costs) and the LR threshold (0.50) are frozen across folds.

## 1 Imports & Setup

In [1]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.insert(0, str(Path.cwd()))
from breakout import (
    LOOKBACK, BB_STD_MULT, KC_ATR_MULT, ATR_WINDOW, ADX_WINDOW, ADX_MIN,
    VOLUME_WINDOW, VOLUME_MULT, STOP_ATR_MULT, PROFIT_ATR_MULT, TIMEOUT_DAYS,
    POSITION_NOTIONAL, STARTING_CAPITAL, RISK_FREE,
    add_indicators, detect_breakouts,
)
from costs import CostConfig, apply_tax, TAX_RATE
from backtest import run_backtest
from metrics import summarize, exit_type_breakdown, drawdown_series, sharpe_ratio
from features import FEATURE_COLS, FeatureBundle, build_feature_row, load_cached
from ml_filter import fit_filter, MIN_PROBABILITY
from hoeffding_monitor import (
    run_monitor, MonitorConfig, hoeffding_bound, effective_N,
    ALERT_AMBER, ALERT_ORANGE, ALERT_RED,
)
from fetch_data import TICKER_TO_SECTOR

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

OUT_DIR = Path.cwd()
DATA_DIR = OUT_DIR / 'data'
print('HW5 working dir:', OUT_DIR, '  data cache:', DATA_DIR.exists())

HW5 working dir: /home/mht120/projects/FinTech533/FinTech533/Homeworks/HW5   data cache: True


## 2 Strategy parameters (single source of truth, frozen across all folds)

In [2]:
params = {
    'LOOKBACK (BB + KC)':     LOOKBACK,
    'BB_STD_MULT':            BB_STD_MULT,
    'KC_ATR_MULT':            KC_ATR_MULT,
    'ATR_WINDOW':             ATR_WINDOW,
    'ADX_WINDOW':             ADX_WINDOW,
    'ADX_MIN':                ADX_MIN,
    'VOLUME_WINDOW':          VOLUME_WINDOW,
    'VOLUME_MULT':            VOLUME_MULT,
    'STOP_ATR_MULT':          STOP_ATR_MULT,
    'PROFIT_ATR_MULT':        PROFIT_ATR_MULT,
    'TIMEOUT_DAYS':           TIMEOUT_DAYS,
    'POSITION_NOTIONAL':      POSITION_NOTIONAL,
    'STARTING_CAPITAL':       STARTING_CAPITAL,
    'RISK_FREE':              RISK_FREE,
    'TAX_RATE':               TAX_RATE,
    'MIN_PROBABILITY (LR)':   MIN_PROBABILITY,
}
pd.Series(params, name='value').to_frame()

,value
LOOKBACK (BB + KC),20.0000
BB_STD_MULT,2.0000
KC_ATR_MULT,1.5000
ATR_WINDOW,14.0000
ADX_WINDOW,14.0000
ADX_MIN,20.0000
VOLUME_WINDOW,20.0000
VOLUME_MULT,1.3000
STOP_ATR_MULT,2.0000
PROFIT_ATR_MULT,4.0000


## 3 Universe (survivorship-clean, 16 names) and cached data

In [3]:
universe = pd.DataFrame([
    ('NVDA','AI / semis','GPU platform for AI training'),
    ('AVGO','AI / semis','Custom AI silicon + networking'),
    ('SMCI','AI / semis','AI server OEM'),
    ('AMD','AI / semis','AI accelerator challenger'),
    ('VST','Nuclear','Nuclear utility + AI data-center power'),
    ('LEU','Nuclear','Centrus Energy, nuclear fuel'),
    ('MSTR','Bitcoin proxies','Bitcoin treasury company'),
    ('MARA','Bitcoin proxies','Bitcoin miner'),
    ('RIOT','Bitcoin proxies','Bitcoin miner (Riot Platforms)'),
    ('PLTR','Defense / AI','Gov-AI platform'),
    ('LMT','Defense / AI','Defense prime (Lockheed)'),
    ('RTX','Defense / AI','Aerospace + missiles'),
    ('NOC','Defense / AI','Northrop Grumman'),
    ('LLY','GLP-1','Mounjaro + Zepbound'),
    ('AAPL','Tech mega-cap','Apple'),
    ('MSFT','Tech mega-cap','Microsoft'),
    ('AMZN','Tech mega-cap','Amazon'),
    ('GOOGL','Tech mega-cap','Alphabet'),
    ('META','Tech mega-cap','Meta'),
    ('JPM','Financials','JPMorgan'),
    ('BAC','Financials','Bank of America'),
    ('WFC','Financials','Wells Fargo'),
], columns=['symbol','theme','rationale'])

raw = {}
for sym in universe['symbol']:
    try:
        raw[sym] = load_cached(sym, DATA_DIR)
    except Exception as e:
        print(f'  missing {sym}: {e}')
print(f'loaded {len(raw)} tickers of {len(universe)}')
for s in list(raw)[:5]:
    d = raw[s]
    print(f'  {s}: {len(d)} rows {d["timestamp"].min().date()}..{d["timestamp"].max().date()}')

macro   = {sym: load_cached(sym, DATA_DIR) for sym in ['VIX','VIX3M','TNX','FVX','IRX','TYX','SPY']}
sectors = {sym: load_cached(sym, DATA_DIR) for sym in set(TICKER_TO_SECTOR.values()) if (DATA_DIR/f'{sym}.parquet').exists()}
bundle = FeatureBundle(
    vix=macro['VIX'], vix3m=macro['VIX3M'],
    tnx=macro['TNX'], fvx=macro['FVX'], irx=macro['IRX'], tyx=macro['TYX'],
    spy=macro['SPY'],
    ticker_prices=raw, sector_prices=sectors, ticker_to_sector=TICKER_TO_SECTOR,
)
universe


loaded 22 tickers of 22
  NVDA: 1126 rows 2021-01-04..2026-04-17
  AVGO: 1129 rows 2021-01-04..2026-04-17
  SMCI: 1129 rows 2021-01-04..2026-04-17
  AMD: 1126 rows 2021-01-04..2026-04-17
  VST: 1129 rows 2021-01-04..2026-04-17


,symbol,theme,rationale
0,NVDA,AI / semis,GPU platform for AI training
1,AVGO,AI / semis,Custom AI silicon + networking
2,SMCI,AI / semis,AI server OEM
3,AMD,AI / semis,AI accelerator challenger
4,VST,Nuclear,Nuclear utility + AI data-center power
5,LEU,Nuclear,"Centrus Energy, nuclear fuel"
6,MSTR,Bitcoin proxies,Bitcoin treasury company
7,MARA,Bitcoin proxies,Bitcoin miner
8,RIOT,Bitcoin proxies,Bitcoin miner (Riot Platforms)
9,PLTR,Defense / AI,Gov-AI platform


## 4 Expanding walk-forward folds

In [4]:
# Expanding window: every fold's training set grows as we move forward in time.
FOLDS = [
    {'id': 1, 'train_years': [2021],                     'test_year': 2022},
    {'id': 2, 'train_years': [2021, 2022],               'test_year': 2023},
    {'id': 3, 'train_years': [2021, 2022, 2023],         'test_year': 2024},
    {'id': 4, 'train_years': [2021, 2022, 2023, 2024],   'test_year': 2025},
]

def year_slice(df, year):
    start = pd.Timestamp(f'{year}-01-01')
    end   = pd.Timestamp(f'{year+1}-01-01')
    return df[(df['timestamp'] >= start) & (df['timestamp'] < end)].reset_index(drop=True)

def multi_year_slice(df, years):
    mask = df['timestamp'] >= pd.Timestamp(f'{min(years)}-01-01')
    mask &= df['timestamp'] < pd.Timestamp(f'{max(years)+1}-01-01')
    return df[mask].reset_index(drop=True)

# Coverage: bars per ticker per year
cov = []
for sym in universe['symbol']:
    for yr in [2021, 2022, 2023, 2024, 2025]:
        cov.append({'symbol': sym, 'year': yr, 'n_bars': len(year_slice(raw[sym], yr))})
cov_df = pd.DataFrame(cov).pivot(index='symbol', columns='year', values='n_bars').fillna(0).astype(int)
cov_df

year,2021,2022,2023,2024,2025
symbol,,,,,
AAPL,250,250,250,249,51
AMD,252,251,250,249,51
AMZN,250,250,250,249,51
AVGO,252,251,250,252,51
BAC,252,251,250,252,51
GOOGL,252,251,250,249,51
JPM,252,251,250,252,51
LEU,252,251,250,252,55
LLY,252,251,250,252,51


## 5 Per-year candidate generation (strict squeeze + vol + ADX)
Run the backtest per ticker per year to get a candidate trade with its realized outcome. Features are attached (exogenous, t-1).

In [5]:
def generate_candidates_year(year):
    rows = []
    for sym in universe['symbol']:
        dsub = year_slice(raw[sym], year)
        if len(dsub) < 30:
            continue
        sig = detect_breakouts(dsub)
        if (sig != 0).sum() == 0:
            continue
        bl, _ = run_backtest(dsub, lookback=LOOKBACK, signals_override=sig)
        if bl.empty:
            continue
        for _, t in bl.iterrows():
            feats = build_feature_row(bundle, sym, t['entry_date'], include_sector_rs=True)
            feats.update({'symbol': sym, 'year': year,
                          'entry_date': t['entry_date'], 'exit_date': t['exit_date'],
                          'direction': t['direction'], 'qty': t['qty'],
                          'entry_price': t['entry_price'], 'exit_price': t['exit_price'],
                          'gross_pnl': t['gross_pnl'], 'commission': t['commission'],
                          'reg_fees': t['reg_fees'], 'slippage': t['slippage'],
                          'borrow_cost': t['borrow_cost'], 'total_cost': t['total_cost'],
                          'net_pnl': t['net_pnl'], 'trade_return': t['trade_return'],
                          'hold_days': t['hold_days'], 'exit_type': t['exit_type'],
                          'good_trade': t['good_trade']})
            rows.append(feats)
    return pd.DataFrame(rows)

year_cands = {y: generate_candidates_year(y) for y in [2021, 2022, 2023, 2024, 2025]}
for y, df in year_cands.items():
    print(f'  {y}: {len(df):>3} strict candidates  win rate {(df["net_pnl"] > 0).mean() if len(df) else float("nan"):.1%}')

  2021:   7 strict candidates  win rate 57.1%
  2022:   5 strict candidates  win rate 20.0%
  2023:   8 strict candidates  win rate 87.5%
  2024:   8 strict candidates  win rate 25.0%
  2025:   0 strict candidates  win rate nan%


## 6 Per-fold BQS refit
BQS recomputed on each fold's (expanding) training years. The 'high-quality breakout names' can shift across folds — that's the regime-drift diagnostic.

In [6]:
def compute_bqs(train_years):
    rows = []
    for sym in universe['symbol']:
        dfull = raw[sym]
        tr = multi_year_slice(dfull, train_years)
        N = len(tr)
        if N < 60:
            continue
        enriched = add_indicators(tr)
        sig = detect_breakouts(tr, volume_mult=0.0, adx_min=0.0).values
        idxs = np.where(sig != 0)[0]
        n_bo = len(idxs)
        if n_bo == 0:
            rows.append({'symbol':sym,'n_breakouts':0,'n_bars':N,'fade_rate':np.nan,'followthrough':np.nan,'bqs':0.0})
            continue
        fades = 0; fwd = []
        closes = enriched['close'].values
        upper  = enriched['kc_upper'].values
        lower  = enriched['kc_lower'].values
        for i in idxs:
            d = sig[i]
            w = slice(i+1, min(i+4, len(closes)))
            if d == 1 and len(closes[w]) and any(closes[w] <= upper[i]): fades += 1
            elif d == -1 and len(closes[w]) and any(closes[w] >= lower[i]): fades += 1
            e = min(i+10, len(closes)-1)
            if e > i: fwd.append(((closes[e]-closes[i])/closes[i]) * d)
        fade_rate = fades / n_bo
        follow = float(np.nanmedian(fwd)) if fwd else 0.0
        rows.append({'symbol':sym,'n_breakouts':n_bo,'n_bars':N,'fade_rate':fade_rate,
                     'followthrough':follow,'bqs':(n_bo/N)*(1-fade_rate)*follow})
    return pd.DataFrame(rows).merge(universe, on='symbol').sort_values('bqs', ascending=False).reset_index(drop=True)

bqs_per_fold = {}
for f in FOLDS:
    bqs_per_fold[f['id']] = compute_bqs(f['train_years'])
    print(f"Fold {f['id']} (train {f['train_years']}): top-5 BQS = {bqs_per_fold[f['id']].head()['symbol'].tolist()}")

# BQS drift table: top-5 per fold
drift = pd.DataFrame({
    f"Fold {f['id']} ({min(f['train_years'])}-{max(f['train_years'])})": bqs_per_fold[f['id']].head(5)['symbol'].tolist() + ['']*(5 - len(bqs_per_fold[f['id']].head(5)))
    for f in FOLDS
}, index=[f'#{i+1}' for i in range(5)])
drift.to_csv(OUT_DIR/'bqs_drift.csv')
drift

Fold 1 (train [2021]): top-5 BQS = ['LEU', 'VST', 'NVDA', 'LLY', 'GOOGL']


Fold 2 (train [2021, 2022]): top-5 BQS = ['LEU', 'MSFT', 'LMT', 'VST', 'WFC']


Fold 3 (train [2021, 2022, 2023]): top-5 BQS = ['MARA', 'RIOT', 'BAC', 'VST', 'MSFT']


Fold 4 (train [2021, 2022, 2023, 2024]): top-5 BQS = ['RIOT', 'RTX', 'VST', 'MSFT', 'BAC']


,Fold 1 (2021-2021),Fold 2 (2021-2022),Fold 3 (2021-2023),Fold 4 (2021-2024)
#1,LEU,LEU,MARA,RIOT
#2,VST,MSFT,RIOT,RTX
#3,NVDA,LMT,BAC,VST
#4,LLY,VST,VST,MSFT
#5,GOOGL,WFC,MSFT,BAC


## 7 Per-fold StandardScaler + Logistic Regression refit
Strict train/test separation: scaler fits on training-fold candidates only, LR trained on training-fold labels only.

In [7]:
fold_lr_coefs = {}
fold_metrics  = {}
fold_train_n  = {}
fold_oos_trades = []

for f in FOLDS:
    train_df = pd.concat([year_cands.get(y, pd.DataFrame()) for y in f['train_years']], ignore_index=True)
    test_df  = year_cands.get(f['test_year'], pd.DataFrame())
    if train_df.empty or test_df.empty:
        print(f"Fold {f['id']}: skipped (train={len(train_df)}, test={len(test_df)})")
        continue
    combined = pd.concat([train_df.assign(split='train'), test_df.assign(split='test')], ignore_index=True)
    combined['label'] = (combined['net_pnl'] > 0).astype(int)
    train_mask = (combined['split'] == 'train').values
    test_mask  = (combined['split'] == 'test').values
    try:
        result = fit_filter(combined, feature_cols=FEATURE_COLS,
                            train_mask=train_mask, test_mask=test_mask,
                            min_probability=MIN_PROBABILITY)
        combined.loc[train_mask, 'lr_prob'] = result.train_probs
        combined.loc[test_mask,  'lr_prob'] = result.test_probs
        combined['lr_approve'] = (combined['lr_prob'] >= MIN_PROBABILITY).astype('Int64')
        fold_lr_coefs[f['id']] = dict(zip(result.coefs['feature'], result.coefs['coefficient']))
        fold_metrics[f['id']]  = dict(train=result.metrics_train, test=result.metrics_test)
    except Exception as e:
        print(f"Fold {f['id']}: LR fit failed ({e}); using raw candidates")
        combined['lr_prob'] = 0.5
        combined['lr_approve'] = 1
        fold_lr_coefs[f['id']] = {c: 0.0 for c in FEATURE_COLS}
        fold_metrics[f['id']]  = {'train':{}, 'test':{}}
    fold_train_n[f['id']] = int(train_mask.sum())
    test_rows = combined[test_mask].copy()
    test_rows['fold_id']   = f['id']
    test_rows['train_years'] = str(f['train_years'])
    test_rows['test_year'] = f['test_year']
    fold_oos_trades.append(test_rows)
    print(f"Fold {f['id']} (train {f['train_years']} n={int(train_mask.sum())}, test {f['test_year']} n={int(test_mask.sum())}): LR fit ok")

oos_trades = pd.concat(fold_oos_trades, ignore_index=True) if fold_oos_trades else pd.DataFrame()
oos_trades = oos_trades.sort_values('entry_date').reset_index(drop=True)
if len(oos_trades):
    oos_trades['trade_id'] = oos_trades['symbol'] + '-' + oos_trades['fold_id'].astype(str) + '-' + (oos_trades.groupby(['fold_id','symbol']).cumcount()+1).astype(str)
print(f'\nTotal OOS trades across all folds: {len(oos_trades)}')
if len(oos_trades):
    print(oos_trades.groupby(['fold_id','test_year']).size().to_string())

Fold 1 (train [2021] n=7, test 2022 n=5): LR fit ok
Fold 2 (train [2021, 2022] n=12, test 2023 n=8): LR fit ok
Fold 3 (train [2021, 2022, 2023] n=20, test 2024 n=8): LR fit ok
Fold 4: skipped (train=28, test=0)

Total OOS trades across all folds: 21
fold_id  test_year
1        2022         5
2        2023         8
3        2024         8


## 8 LR coefficient stability table (features × folds)
Consistent sign and magnitude across folds = signal. Sign flips across folds = noise.

In [8]:
coef_mat = pd.DataFrame(index=FEATURE_COLS)
for fid, coefs in fold_lr_coefs.items():
    coef_mat[f'Fold {fid}'] = [coefs.get(f, np.nan) for f in FEATURE_COLS]
coef_mat = coef_mat.round(3)
coef_mat.to_csv(OUT_DIR/'lr_coef_stability.csv')
coef_mat

,Fold 1,Fold 2,Fold 3
curve_a3,-0.250,-0.194,-0.283
curve_a2,0.164,0.124,0.204
curve_a1,0.681,0.147,0.018
curve_a0,-0.834,-0.495,-0.170
vix_level,-0.025,-0.034,-0.067
vix3m_level,0.033,0.120,0.028
vix_spread,0.183,0.391,0.377
vix_change_20d,0.242,0.283,-0.009
iv_level,-0.025,-0.034,-0.067
rv_level,0.207,0.203,0.337


## 9 Stitched OOS equity curve (pool the 4 folds into one continuous line)

In [9]:
def build_signal_from_candidates(df_slice, sym, cands_df):
    sig = pd.Series(0, index=df_slice.index, dtype=int)
    if cands_df is None or cands_df.empty:
        return sig
    cd = cands_df[cands_df['symbol'] == sym].copy()
    if cd.empty: return sig
    cd['entry_date'] = pd.to_datetime(cd['entry_date']).dt.date
    # LR is annotated but NOT gating in this run; take every strict signal so the
    # ledger equity curve matches the blotter trade count.
    approved = set(cd['entry_date'])
    dates = pd.to_datetime(df_slice['timestamp']).dt.date.values
    strict = detect_breakouts(df_slice).values
    for i in range(len(sig)):
        if strict[i] == 0: continue
        if i+1 >= len(dates): continue
        if dates[i+1] in approved:
            sig.iloc[i] = strict[i]
    return sig

all_ledgers = []
all_blotters_rerun = []
for f in FOLDS:
    fold_trades_df = oos_trades[oos_trades['fold_id'] == f['id']]
    for sym in universe['symbol']:
        dsub = year_slice(raw[sym], f['test_year'])
        if len(dsub) < 30:
            continue
        sig = build_signal_from_candidates(dsub, sym, fold_trades_df)
        if (sig != 0).sum() == 0:
            continue
        bl, lg = run_backtest(dsub, lookback=LOOKBACK, signals_override=sig)
        if not lg.empty:
            all_ledgers.append(lg.assign(symbol=sym, fold_id=f['id'], test_year=f['test_year']))
        if not bl.empty:
            all_blotters_rerun.append(bl.assign(symbol=sym, fold_id=f['id']))

stitched_ledger = pd.concat(all_ledgers, ignore_index=True) if all_ledgers else pd.DataFrame()
stitched_blotter = pd.concat(all_blotters_rerun, ignore_index=True) if all_blotters_rerun else pd.DataFrame()
print(f'Stitched ledger rows: {len(stitched_ledger)}   stitched blotter rows: {len(stitched_blotter)}')

Stitched ledger rows: 4517   stitched blotter rows: 21


## 10 Export blotter & ledger

In [10]:
blotter_cols = [
    'trade_id','symbol','fold_id','train_years','test_year',
    'entry_date','exit_date','direction','qty','entry_price','exit_price',
    'gross_pnl','commission','reg_fees','slippage','borrow_cost','total_cost',
    'net_pnl','trade_return','hold_days','exit_type','good_trade',
    'lr_prob','lr_approve',
]
for c in blotter_cols:
    if c not in oos_trades.columns:
        oos_trades[c] = np.nan
oos_trades[blotter_cols].to_csv(OUT_DIR/'trades.csv', index=False)
stitched_ledger.to_csv(OUT_DIR/'ledger.csv', index=False)
print(f'trades.csv ({len(oos_trades)} rows), ledger.csv ({len(stitched_ledger)} rows)')
oos_trades.head()

trades.csv (21 rows), ledger.csv (4517 rows)


,curve_a3,curve_a2,curve_a1,curve_a0,vix_level,vix3m_level,vix_spread,vix_change_20d,iv_level,rv_level,iv_rv_spread,spy_ret_20d,sector_rs,sector_ret_5d,symbol,year,entry_date,exit_date,direction,qty,entry_price,exit_price,gross_pnl,commission,reg_fees,slippage,borrow_cost,total_cost,net_pnl,trade_return,hold_days,exit_type,good_trade,split,label,lr_prob,lr_approve,fold_id,train_years,test_year,trade_id
0,0.001067,-0.048077,0.551284,0.205167,31.02,30.88,-0.14,1.12,31.02,20.801074,10.218926,-0.028817,-0.133159,-0.033992,LEU,2022,2022-02-24,2022-02-28,short,280,35.682150,42.789425,-1990.037045,1.960,0.324230,10.986021,2.378810,15.649060,-2005.686105,-0.200749,2,stop_loss,0,test,0,4.570275e-01,0,1,[2021],2022,LEU-1-1
1,0.001076,-0.048603,0.560960,0.160781,27.59,28.04,0.45,-0.07,27.59,22.318140,5.271860,0.015096,0.033976,0.019320,LMT,2022,2022-02-28,2022-03-01,long,24,414.016905,448.340235,823.759929,0.700,0.303117,10.348286,0.000000,11.351402,812.408527,0.081761,1,profit_target,1,test,1,9.919966e-01,1,1,[2021],2022,LMT-1-1
2,0.000950,-0.043009,0.502486,0.165052,30.15,29.65,-0.50,2.49,30.15,22.053844,8.096156,-0.012038,-0.050116,-0.017848,JPM,2022,2022-03-01,2022-03-29,short,71,139.969980,141.180000,-85.911420,0.700,0.288059,9.980824,23.661592,34.630475,-120.541895,-0.012130,20,timeout,0,test,0,7.709994e-01,1,1,[2021],2022,JPM-1-1
3,0.001501,-0.068059,0.777389,0.609883,34.75,35.05,0.30,13.59,34.75,27.129108,7.620892,-0.110374,-0.164655,-0.120961,LEU,2022,2022-05-10,2022-06-07,short,454,21.998995,28.859809,-3114.809475,3.178,0.353018,11.544948,22.590873,37.666839,-3152.476314,-0.315641,19,stop_loss,0,test,0,1.558799e-06,0,1,[2021],2022,LEU-1-2
4,0.000270,-0.010624,0.081938,3.910175,27.39,28.28,0.89,-5.21,27.39,26.291099,1.098901,0.047383,-0.280813,0.025434,META,2022,2022-10-28,2022-11-14,short,100,99.520215,116.247284,-1672.706921,0.700,0.293266,10.788375,13.032409,24.814050,-1697.520972,-0.170570,11,stop_loss,0,test,0,4.760403e-87,0,1,[2021],2022,META-1-1


## 11 Per-fold and pooled OOS metrics

In [11]:
def metrics_row(trades_df, ledger_df=None, label=''):
    if len(trades_df) == 0:
        return {'label': label, 'n_trades': 0, 'win_rate': np.nan, 'avg_return': np.nan,
                'net_pnl_usd': 0.0, 'sharpe': np.nan, 'max_dd': np.nan, 'profit_factor': np.nan}
    lg = ledger_df if ledger_df is not None else pd.DataFrame({'mkt_value':[STARTING_CAPITAL],'daily_return':[0.0]})
    m = summarize(trades_df, lg)
    return {'label': label,
            'n_trades': m.get('n_trades', 0),
            'win_rate': m.get('win_rate', np.nan),
            'avg_return': m.get('avg_return_per_trade', np.nan),
            'net_pnl_usd': m.get('total_net_pnl_usd', 0.0),
            'sharpe': m.get('sharpe', np.nan),
            'max_dd': m.get('max_drawdown', np.nan),
            'profit_factor': m.get('profit_factor', np.nan)}

rows = []
for f in FOLDS:
    ft = oos_trades[oos_trades['fold_id'] == f['id']]
    fl = stitched_ledger[stitched_ledger['fold_id'] == f['id']] if 'fold_id' in stitched_ledger.columns else pd.DataFrame()
    rows.append(metrics_row(ft, fl if not fl.empty else None, label=f"Fold {f['id']}: train {min(f['train_years'])}-{max(f['train_years'])} -> test {f['test_year']}"))
rows.append(metrics_row(oos_trades, stitched_ledger, label='Pooled OOS'))
fold_metrics_df = pd.DataFrame(rows)
fold_metrics_df.to_csv(OUT_DIR/'fold_metrics.csv', index=False)
fold_metrics_df

,label,n_trades,win_rate,avg_return,net_pnl_usd,sharpe,max_dd,profit_factor
0,Fold 1: train 2021-2021 -> test 2022,5,0.20000,-0.123466,-6163.816759,-2.664860,-0.051582,0.116454
1,Fold 2: train 2021-2022 -> test 2023,8,0.87500,0.104688,8377.898487,-1.033745,-0.037375,19.927998
2,Fold 3: train 2021-2023 -> test 2024,8,0.25000,-0.035287,-2776.306110,-6.544684,-0.012814,0.231128
3,Fold 4: train 2021-2024 -> test 2025,0,NaN,NaN,0.000000,NaN,NaN,NaN
4,Pooled OOS,21,0.47619,-0.002958,-562.224382,-2.280744,-0.051582,0.949026


## 12 Hoeffding Regime Monitor on stitched OOS stream

In [12]:
# Baselines come from the UNION of all training years
all_train_years = sorted({y for f in FOLDS for y in f['train_years']})
baseline_df = pd.concat([year_cands[y] for y in all_train_years if y in year_cands], ignore_index=True)
mu_W_train = float(baseline_df['good_trade'].mean()) if len(baseline_df) else 0.5
mu_R_train = float(baseline_df['trade_return'].mean()) if len(baseline_df) else 0.0
R_RANGE_FRACTION = 0.08
cfg = MonitorConfig(mu_W=mu_W_train, mu_R=mu_R_train, R_range=R_RANGE_FRACTION)
print(f'Baselines  mu_W = {mu_W_train:.3f}   mu_R = {mu_R_train:+.4f}   R_range = {R_RANGE_FRACTION}')
monitor_df = run_monitor(oos_trades.sort_values('entry_date').reset_index(drop=True), cfg) if len(oos_trades) else pd.DataFrame()
monitor_df.to_csv(OUT_DIR/'hoeffding_monitor.csv', index=False)
monitor_df.tail(15)

Baselines  mu_W = 0.500   mu_R = +0.0002   R_range = 0.08


,trade_n,Xbar_W,t_W,P_W,Xbar_R,t_R,P_R,rho,N_eff,P_min,alert
6,7,0.428571,0.071429,0.931063,-0.041621,0.041846,0.021697,0.000000,7.000000,0.021697,RED
7,8,0.500000,0.000000,1.000000,-0.021421,0.021646,0.433140,0.166667,5.714286,0.433140,AMBER
8,9,0.555556,0.000000,1.000000,-0.012597,0.012823,0.761369,0.258199,5.306164,0.761369,GREEN
9,10,0.600000,0.000000,1.000000,0.018743,0.000000,1.000000,0.316228,5.194939,1.000000,GREEN
10,11,0.545455,0.000000,1.000000,0.012921,0.000000,1.000000,0.166667,7.857143,1.000000,GREEN
11,12,0.583333,0.000000,1.000000,0.017174,0.000000,1.000000,0.069007,10.450751,1.000000,GREEN
12,13,0.615385,0.000000,1.000000,0.016937,0.000000,1.000000,0.119523,10.224180,1.000000,GREEN
13,14,0.571429,0.000000,1.000000,0.009859,0.000000,1.000000,0.025000,13.317073,1.000000,GREEN
14,15,0.533333,0.000000,1.000000,0.008198,0.000000,1.000000,0.125000,11.666667,1.000000,GREEN
15,16,0.500000,0.000000,1.000000,0.004320,0.000000,1.000000,0.196429,10.746269,1.000000,GREEN


## 13 Exit Attribution (every OOS trade)

In [13]:
if len(oos_trades) > 0:
    ex = exit_type_breakdown(oos_trades)
    color_map = {'profit_target': '#2e7d32', 'stop_loss': '#c62828', 'timeout': '#f9a825'}
    fig = go.Figure(go.Pie(
        labels=ex['exit_type'], values=ex['count'], hole=0.55,
        marker=dict(colors=[color_map.get(x, '#999') for x in ex['exit_type']]),
        textinfo='label+percent+value', textposition='outside',
    ))
    total = int(ex['count'].sum())
    fig.update_layout(title='Exit Attribution across all OOS folds',
                      template='simple_white', height=420,
                      annotations=[dict(text=f'{total}<br>trades', x=0.5, y=0.5, showarrow=False, font=dict(size=18))])
    fig.write_html(str(OUT_DIR/'exit_attribution.html'), include_plotlyjs='cdn')
    fig.show()
    ex

## 14 Three-line Equity: Gross vs Net vs After-Tax (stitched OOS)

In [14]:
if not stitched_ledger.empty:
    lc = stitched_ledger.copy()
    lc['date'] = pd.to_datetime(lc['date'])
    lc['delta'] = lc['mkt_value'] - STARTING_CAPITAL
    agg = lc.groupby('date')['delta'].sum().reset_index()
    agg['net_value'] = STARTING_CAPITAL + agg['delta']
    if not oos_trades.empty:
        cb = oos_trades.groupby(pd.to_datetime(oos_trades['exit_date']))['total_cost'].sum()
    else:
        cb = pd.Series(dtype=float)
    cum, gross = 0.0, []
    for d in agg['date']:
        cum += cb.get(d, 0.0); gross.append(cum)
    agg['cum_costs'] = gross
    agg['gross_value'] = agg['net_value'] + agg['cum_costs']
    agg['net_pnl_cum'] = agg['net_value'] - STARTING_CAPITAL
    agg['after_tax_pnl'] = agg['net_pnl_cum'].apply(lambda p: p * (1 - TAX_RATE) if p > 0 else p)
    agg['after_tax_value'] = STARTING_CAPITAL + agg['after_tax_pnl']
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=agg['date'], y=agg['gross_value'],     line=dict(color='steelblue'), name='Gross PnL'))
    fig.add_trace(go.Scatter(x=agg['date'], y=agg['net_value'],       line=dict(color='navy'),      name='Net of friction'))
    fig.add_trace(go.Scatter(x=agg['date'], y=agg['after_tax_value'], line=dict(color='darkorange'),name='After 30% cap-gains tax'))
    fig.add_hline(y=STARTING_CAPITAL, line_dash='dot', line_color='gray')
    # Add vertical fold separators
    for f in FOLDS[1:]:
        fig.add_vline(x=pd.Timestamp(f'{f["test_year"]}-01-01'), line_color='lightgray', line_dash='dot')
    fig.update_layout(title='Stitched OOS 2022-2025: Gross vs Net vs After-Tax',
                      template='simple_white', height=480, yaxis_title='Portfolio value ($)')
    fig.write_html(str(OUT_DIR/'equity_3line.html'), include_plotlyjs='cdn')
    fig.show()

## 15 Drawdown (stitched OOS)

In [15]:
if not stitched_ledger.empty:
    dd = drawdown_series(agg['net_value'])
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=agg['date'], y=dd, fill='tozeroy', line=dict(color='crimson'), name='Drawdown'))
    for f in FOLDS[1:]:
        fig.add_vline(x=pd.Timestamp(f'{f["test_year"]}-01-01'), line_color='lightgray', line_dash='dot')
    fig.update_layout(title='Stitched OOS drawdown (underwater curve)',
                      template='simple_white', height=360, yaxis=dict(tickformat='.1%'))
    fig.write_html(str(OUT_DIR/'drawdown.html'), include_plotlyjs='cdn')
    fig.show()

## 16 Hoeffding monitor plot (50/25/10% threshold lines)

In [16]:
if len(monitor_df) > 0:
    fig = make_subplots(specs=[[{'secondary_y': True}]])
    fig.add_trace(go.Scatter(x=monitor_df['trade_n'], y=monitor_df['P_W'],   name='P(W) win-rate',     line=dict(color='royalblue', width=2)), secondary_y=False)
    fig.add_trace(go.Scatter(x=monitor_df['trade_n'], y=monitor_df['P_R'],   name='P(R) return',        line=dict(color='firebrick', width=2)), secondary_y=False)
    fig.add_trace(go.Scatter(x=monitor_df['trade_n'], y=monitor_df['P_min'], name='P_min',              line=dict(color='black', width=1, dash='dash')), secondary_y=False)
    fig.add_trace(go.Bar    (x=monitor_df['trade_n'], y=monitor_df['N_eff'], name='N_eff',              marker_color='lightgray', opacity=0.5), secondary_y=True)
    fig.add_hline(y=ALERT_AMBER,  line_dash='dot', line_color='goldenrod', annotation_text=f'{ALERT_AMBER:.0%} amber')
    fig.add_hline(y=ALERT_ORANGE, line_dash='dot', line_color='orangered', annotation_text=f'{ALERT_ORANGE:.0%} orange')
    fig.add_hline(y=ALERT_RED,    line_dash='dot', line_color='red',       annotation_text=f'{ALERT_RED:.0%} HALT')
    fig.update_yaxes(title_text='Hoeffding upper bound (probability)', range=[0, 1.05], secondary_y=False)
    fig.update_yaxes(title_text='N_eff', secondary_y=True, showgrid=False)
    fig.update_layout(title='Hoeffding Regime Monitor, OOS stream (all folds)', template='simple_white', height=520,
                      xaxis_title='Cumulative trade # (chronological)')
    fig.write_html(str(OUT_DIR/'hoeffding_plot.html'), include_plotlyjs='cdn')
    fig.show()

## 17 Slippage sensitivity on stitched OOS

In [17]:
slip_rows = []
for bps in [5, 10, 15]:
    cfg_s = CostConfig(slippage_bps=bps)
    bl_all, lg_all = [], []
    for f in FOLDS:
        fold_trades_df = oos_trades[oos_trades['fold_id'] == f['id']]
        for sym in universe['symbol']:
            dsub = year_slice(raw[sym], f['test_year'])
            if len(dsub) < 30: continue
            sig = build_signal_from_candidates(dsub, sym, fold_trades_df)
            if (sig != 0).sum() == 0: continue
            bl, lg = run_backtest(dsub, lookback=LOOKBACK, signals_override=sig, cost_cfg=cfg_s)
            if not bl.empty: bl_all.append(bl.assign(symbol=sym))
            if not lg.empty: lg_all.append(lg)
    b_df = pd.concat(bl_all, ignore_index=True) if bl_all else pd.DataFrame()
    l_df = pd.concat(lg_all, ignore_index=True) if lg_all else pd.DataFrame()
    mt = summarize(b_df, l_df) if len(b_df) else {}
    slip_rows.append({
        'slippage_bps': bps, 'n_trades': mt.get('n_trades', 0),
        'net_pnl_usd': mt.get('total_net_pnl_usd', 0.0),
        'avg_return_per_trade': mt.get('avg_return_per_trade'),
        'sharpe': mt.get('sharpe'),
        'cost_pct_of_gross': mt.get('cost_pct_of_gross'),
    })
slip_sens = pd.DataFrame(slip_rows)
slip_sens.to_csv(OUT_DIR/'slippage_sensitivity.csv', index=False)
slip_sens

,slippage_bps,n_trades,net_pnl_usd,avg_return_per_trade,sharpe,cost_pct_of_gross
0,5,21,-562.224382,-0.002958,-2.280744,1.702049
1,10,21,-806.511032,-0.004165,-2.284273,2.391862
2,15,21,-1050.856241,-0.005371,-2.292146,2.928682


## 18 Discussion

**Expanding walk-forward.** Four folds, each refitting BQS + StandardScaler + Logistic Regression on the growing training history. The strategy design parameters (BB/KC lookback, ATR multiples, exits, position sizing, cost model) and the LR threshold (0.50) are frozen across all folds so no hyperparameters leak forward.

**Survivorship discipline.** Sixteen names that traded continuously from 2021-01-01. Post-2021 IPOs (OKLO, SMR, RGTI, IONQ, QBTS, COIN, RKLB) excluded to avoid the classic overfitting failure mode where the AI infers patterns from names that weren't publicly investable at the fold's training boundary.

**Signal kept, secondary confirmations kept.** Keltner Squeeze + volume (>= 1.3x) + ADX(14) >= 20. Vestal's interpretation stands: low-volume squeezes are traps, low-ADX squeezes are fakeouts.

**What the outputs show.** The LR coefficient stability table reveals which exogenous features carry consistent sign across folds (signal) versus which flip sign (noise). The BQS drift table shows whether the top-5 breakout-quality names persist across training expansions or churn annually. The Hoeffding monitor runs on the full chronological OOS stream (~N trades pooled across 4 folds), so its 1/sqrt(N) convergence produces meaningfully tighter bounds than a one-fold run.

**Natural extensions for the final project.** Top-3 BQS portfolio (trade only the top-ranked names in each fold) with correlation-aware sizing. Kelly position sizing conditional on LR probability. Monte-Carlo resampling of the stitched OOS stream to stress-test Sharpe. Regime-adaptive ATR multipliers.